# Tilt stats

A tilt-distance-specific census for anticyclonic (AE) and cyclonic (CE) eddies. The notebook retains the original mirrored-histogram idea but uses compact publication dashboards, smooth shading and embedded weighted box summaries.

Daily distributions give every eddy equal total weight, preventing long tracks from dominating. Per-eddy metrics contain one observation per eddy.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter
import seacofs_tilt_tools as tilt

AE_COLOUR = '#C44E52'
CE_COLOUR = '#2878A5'
TYPE_COLOURS = {'AE': AE_COLOUR, 'CE': CE_COLOUR}
PAGE_SIZE = (10, 7)
SAVE_FIGURES = False
FIGURE_DIR = Path('tilt_census_figures')
SMOOTH_SIGMA_BINS = 1.15
UPSTREAM_REGIONS = ['S1', 'U1', 'U2']
DOWNSTREAM_REGIONS = ['S2', 'D1', 'D2']
SUBREGION_ORDER = ['S1', 'U1', 'U2', 'S2', 'D1', 'D2']

mpl.rcParams.update({'figure.dpi':120,'savefig.dpi':600,'font.size':9.5,
 'axes.titlesize':10.5,'axes.labelsize':9.5,'legend.fontsize':8.5,
 'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.8,
 'xtick.direction':'out','ytick.direction':'out','pdf.fonttype':42,'ps.fonttype':42})

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths, add_regions=True, grid=grid)
df_eddies = df_eddies.sort_values(['Cyc','Eddy','Day']).copy()
region_group_map = {'S1':'Shelf','S2':'Shelf','U1':'Upstream','U2':'Upstream','D1':'Downstream','D2':'Downstream'}
sector_map = {**{r:'Upstream + S1' for r in UPSTREAM_REGIONS}, **{r:'Downstream + S2' for r in DOWNSTREAM_REGIONS}}
df_eddies['RegionGroup'] = df_eddies.Region.map(region_group_map)
df_eddies['Sector'] = df_eddies.Region.map(sector_map)
KEYS = ['Cyc','Eddy']
df_eddies['day_index'] = df_eddies.groupby(KEYS).cumcount()
last_index = df_eddies.groupby(KEYS).day_index.transform('max')
df_eddies['norm_time'] = np.where(last_index>0,df_eddies.day_index/last_index,np.nan)
df_eddies['lifetime_days'] = df_eddies.groupby(KEYS).Day.transform(lambda x:x.max()-x.min()+1)
tilt_data = df_eddies.dropna(subset=['TiltDis']).query("Cyc in ['AE','CE']").copy()
# Tilt distance and Rc are both in km, so this ratio is dimensionless.
valid_rc = np.isfinite(tilt_data['Rc']) & (tilt_data['Rc'] > 0)
tilt_data['tilt_over_Rc'] = np.where(valid_rc, tilt_data['TiltDis'] / tilt_data['Rc'], np.nan)
GLOBAL_TILT_LIMIT = float(tilt_data.TiltDis.quantile(.995))
GLOBAL_TILT_BINS = np.linspace(0,GLOBAL_TILT_LIMIT,43)
GLOBAL_TILT_RC_BINS = np.linspace(0,tilt_data.tilt_over_Rc.quantile(.995),43)
df_eddies.head()

,Eddy,Day,Cyc,lon,lat,ic,jc,xc,yc,w,...,Date,fname,TiltDis,TiltDir,Region,RegionGroup,Sector,day_index,norm_time,lifetime_days
26,2,1462,AE,151.793573,-37.940740,140,40,367.413413,196.922965,0.000038,...,1994-01-02,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,NaN,NaN,D1,Downstream,Downstream + S2,0,0.000000,32
27,2,1463,AE,151.615529,-38.053390,136,36,356.354073,179.836500,0.000042,...,1994-01-03,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,NaN,NaN,D1,Downstream,Downstream + S2,1,0.032258,32
28,2,1464,AE,151.505429,-38.086839,133,35,348.312905,173.055772,0.000041,...,1994-01-04,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,NaN,NaN,D1,Downstream,Downstream + S2,2,0.064516,32
29,2,1465,AE,151.720810,-38.058377,139,37,365.294728,182.464807,0.000036,...,1994-01-05,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,11.672594,226.828579,D1,Downstream,Downstream + S2,3,0.096774,32
30,2,1466,AE,151.590538,-38.149537,136,34,357.511090,169.056926,0.000035,...,1994-01-06,/srv/scratch/z3533156/26year_BRAN2020/outer_av...,12.557221,221.266550,D1,Downstream,Downstream + S2,4,0.129032,32


### Paper tilt statistics

In [7]:
from IPython.display import display, Markdown

def tilt_census_report(
    df,
    first_last_days=3,
    upstream=("S1", "U1", "U2"),
    downstream=("S2", "D1", "D2"),
):
    keys = ["Cyc", "Eddy"]

    d = (
        df[df["Cyc"].isin(["AE", "CE"])]
        .sort_values(keys + ["Day"])
        .copy()
    )

    # Explicit Delta-method eligibility.
    d["track_index"] = d.groupby(keys).cumcount()
    d["track_length"] = d.groupby(keys)["Day"].transform("size")
    d["tilt_possible"] = (
        (d["track_index"] >= first_last_days)
        & (d["track_index"] < d["track_length"] - first_last_days)
    )
    d["valid_tilt"] = d["TiltDis"].notna()

    # Dimensionless daily tilt relative to surface radius.
    valid_rc = np.isfinite(d["Rc"]) & (d["Rc"] > 0)
    d["tilt_over_Rc"] = np.where(
        valid_rc,
        d["TiltDis"] / d["Rc"],
        np.nan,
    )

    d["Sector"] = pd.NA
    d.loc[d["Region"].isin(upstream), "Sector"] = "Upstream + S1"
    d.loc[d["Region"].isin(downstream), "Sector"] = "Downstream + S2"

    valid = d[d["valid_tilt"]].copy()

    # Dataset census.
    census_rows = []
    for cyc in ["All", "AE", "CE"]:
        g = d if cyc == "All" else d[d["Cyc"] == cyc]

        possible = int(g["tilt_possible"].sum())
        valid_possible = int(
            (g["valid_tilt"] & g["tilt_possible"]).sum()
        )

        census_rows.append({
            "Cyc": cyc,
            "unique_eddies": g[keys].drop_duplicates().shape[0],
            "eddy_days": len(g),
            "possible_tilt_days": possible,
            "valid_tilt_days": int(g["valid_tilt"].sum()),
            "raw_coverage_percent":
                100 * g["valid_tilt"].mean(),
            "possible_coverage_percent":
                100 * valid_possible / possible
                if possible else np.nan,
        })

    census = pd.DataFrame(census_rows).set_index("Cyc")

    # Daily tilt statistics.
    daily_rows = []
    for cyc in ["All", "AE", "CE"]:
        g = valid if cyc == "All" else valid[valid["Cyc"] == cyc]
        x = g["TiltDis"].dropna()
        r = g["tilt_over_Rc"].dropna()

        counts, edges = np.histogram(
            x,
            bins=np.arange(0, max(205, x.max() + 5), 5),
        )
        peak = np.argmax(counts)

        daily_rows.append({
            "Cyc": cyc,
            "n": len(x),
            "mean_km": x.mean(),
            "median_km": x.median(),
            "q25_km": x.quantile(.25),
            "q75_km": x.quantile(.75),
            "q90_km": x.quantile(.90),
            "q95_km": x.quantile(.95),
            "q99_km": x.quantile(.99),
            "maximum_km": x.max(),
            "modal_5km_interval":
                f"{edges[peak]:.0f}–{edges[peak + 1]:.0f}",
            "percent_gt_10km": 100 * (x > 10).mean(),
            "percent_gt_20km": 100 * (x > 20).mean(),
            "percent_gt_50km": 100 * (x > 50).mean(),
            "percent_gt_100km": 100 * (x > 100).mean(),
            "median_tilt_over_Rc": r.median(),
            "percent_tilt_gt_Rc": 100 * (r > 1).mean(),
        })

    daily = pd.DataFrame(daily_rows).set_index("Cyc")

    # One row per eddy.
    eddy_rows = []
    for (cyc, eddy), g in valid.groupby(keys):
        x = g["TiltDis"].dropna()
        r = g["tilt_over_Rc"].dropna()

        eddy_rows.append({
            "Cyc": cyc,
            "Eddy": eddy,
            "n_tilt": len(x),
            "mean_tilt_km": x.mean(),
            "median_tilt_km": x.median(),
            "minimum_tilt_km": x.min(),
            "maximum_tilt_km": x.max(),
            "range_tilt_km": x.max() - x.min(),
            "mean_tilt_over_Rc":
                r.mean() if len(r) else np.nan,
        })

    eddies = pd.DataFrame(eddy_rows)

    eddy_summary = (
        eddies.groupby("Cyc")
        .agg(
            eddies=("Eddy", "size"),
            median_eddy_mean_km=("mean_tilt_km", "median"),
            median_eddy_median_km=("median_tilt_km", "median"),
            median_eddy_minimum_km=("minimum_tilt_km", "median"),
            median_eddy_maximum_km=("maximum_tilt_km", "median"),
            median_eddy_range_km=("range_tilt_km", "median"),
            median_mean_tilt_over_Rc=(
                "mean_tilt_over_Rc", "median"
            ),
            percent_min_below_10km=(
                "minimum_tilt_km",
                lambda x: 100 * (x < 10).mean(),
            ),
            percent_max_above_50km=(
                "maximum_tilt_km",
                lambda x: 100 * (x > 50).mean(),
            ),
            percent_max_above_100km=(
                "maximum_tilt_km",
                lambda x: 100 * (x > 100).mean(),
            ),
            percent_max_above_200km=(
                "maximum_tilt_km",
                lambda x: 100 * (x > 200).mean(),
            ),
            largest_eddy_maximum_km=(
                "maximum_tilt_km", "max"
            ),
        )
    )

    # Sector comparison.
    sector = (
        valid.dropna(subset=["Sector"])
        .groupby(["Cyc", "Sector"])
        .agg(
            tilt_estimates=("TiltDis", "size"),
            mean_tilt_km=("TiltDis", "mean"),
            median_tilt_km=("TiltDis", "median"),
            q25_tilt_km=(
                "TiltDis",
                lambda x: x.quantile(.25),
            ),
            q75_tilt_km=(
                "TiltDis",
                lambda x: x.quantile(.75),
            ),
            median_tilt_over_Rc=(
                "tilt_over_Rc", "median"
            ),
            percent_gt_50km=(
                "TiltDis",
                lambda x: 100 * (x > 50).mean(),
            ),
        )
    )

    # Exact latitude bands used in the manuscript.
    valid["LatitudeZone"] = "32–36°S"
    valid.loc[valid["lat"] > -32, "LatitudeZone"] = "North of 32°S"
    valid.loc[valid["lat"] < -36, "LatitudeZone"] = "South of 36°S"

    latitude = (
        valid.groupby(["Cyc", "LatitudeZone"])
        .agg(
            n=("TiltDis", "size"),
            mean_tilt_km=("TiltDis", "mean"),
            median_tilt_km=("TiltDis", "median"),
            q25_tilt_km=(
                "TiltDis",
                lambda x: x.quantile(.25),
            ),
            q75_tilt_km=(
                "TiltDis",
                lambda x: x.quantile(.75),
            ),
        )
    )

    # Short, directly copyable results.
    ae_c, ce_c, all_c = (
        census.loc["AE"],
        census.loc["CE"],
        census.loc["All"],
    )
    ae_d, ce_d, all_d = (
        daily.loc["AE"],
        daily.loc["CE"],
        daily.loc["All"],
    )
    ae_e, ce_e = eddy_summary.loc["AE"], eddy_summary.loc["CE"]

    sentences = [
        f"Total eddies: {all_c.unique_eddies:,.0f}.",
        f"AEs: {ae_c.unique_eddies:,.0f}; "
        f"CEs: {ce_c.unique_eddies:,.0f}.",

        f"Total eddy-days: {all_c.eddy_days:,.0f}.",
        f"AE-days: {ae_c.eddy_days:,.0f}; "
        f"CE-days: {ce_c.eddy_days:,.0f}.",

        f"Valid tilt estimates: "
        f"{all_c.valid_tilt_days:,.0f} "
        f"({all_c.raw_coverage_percent:.1f}% of all eddy-days).",

        f"Possible-estimate success: "
        f"{ae_c.possible_coverage_percent:.1f}% for AEs and "
        f"{ce_c.possible_coverage_percent:.1f}% for CEs.",

        f"Mean AE tilt distance: {ae_d.mean_km:.1f} km.",
        f"Mean CE tilt distance: {ce_d.mean_km:.1f} km.",

        f"Median AE tilt distance: {ae_d.median_km:.1f} km "
        f"(IQR {ae_d.q25_km:.1f}–{ae_d.q75_km:.1f} km).",

        f"Median CE tilt distance: {ce_d.median_km:.1f} km "
        f"(IQR {ce_d.q25_km:.1f}–{ce_d.q75_km:.1f} km).",

        f"Modal 5-km tilt interval: "
        f"{ae_d.modal_5km_interval} km for AEs and "
        f"{ce_d.modal_5km_interval} km for CEs.",

        f"Tilt exceeded 50 km on "
        f"{ae_d.percent_gt_50km:.1f}% of AE-days and "
        f"{ce_d.percent_gt_50km:.1f}% of CE-days.",

        f"Tilt exceeded 100 km on "
        f"{ae_d.percent_gt_100km:.1f}% of AE-days and "
        f"{ce_d.percent_gt_100km:.1f}% of CE-days.",

        f"Median tilt/Rc: {ae_d.median_tilt_over_Rc:.2f} for AEs "
        f"and {ce_d.median_tilt_over_Rc:.2f} for CEs.",

        f"Tilt exceeded one Rc on "
        f"{ae_d.percent_tilt_gt_Rc:.1f}% of AE-days and "
        f"{ce_d.percent_tilt_gt_Rc:.1f}% of CE-days.",

        f"Median per-eddy mean tilt: "
        f"{ae_e.median_eddy_mean_km:.1f} km for AEs and "
        f"{ce_e.median_eddy_mean_km:.1f} km for CEs.",

        f"At least one tilt above 50 km occurred in "
        f"{ae_e.percent_max_above_50km:.1f}% of AEs and "
        f"{ce_e.percent_max_above_50km:.1f}% of CEs.",

        f"At least one tilt above 100 km occurred in "
        f"{ae_e.percent_max_above_100km:.1f}% of AEs and "
        f"{ce_e.percent_max_above_100km:.1f}% of CEs.",
    ]

    paragraph = (
        f"Application of the Delta method produced "
        f"{all_c.valid_tilt_days:,.0f} valid tilt estimates, "
        f"corresponding to {all_c.raw_coverage_percent:.1f}% of all "
        f"eddy-days. The dataset contained "
        f"{all_c.unique_eddies:,.0f} unique eddies, comprising "
        f"{ae_c.unique_eddies:,.0f} anticyclonic eddies and "
        f"{ce_c.unique_eddies:,.0f} cyclonic eddies. Valid estimates "
        f"were obtained for {ae_c.valid_tilt_days:,.0f} AE-days and "
        f"{ce_c.valid_tilt_days:,.0f} CE-days, representing "
        f"{ae_c.possible_coverage_percent:.1f}% and "
        f"{ce_c.possible_coverage_percent:.1f}% of possible estimates, "
        f"respectively. Median tilt distances were "
        f"{ae_d.median_km:.1f} km for AEs and "
        f"{ce_d.median_km:.1f} km for CEs, with interquartile ranges "
        f"of {ae_d.q25_km:.1f}–{ae_d.q75_km:.1f} km and "
        f"{ce_d.q25_km:.1f}–{ce_d.q75_km:.1f} km, respectively. "
        f"Tilt exceeded 50 km on {ae_d.percent_gt_50km:.1f}% of "
        f"AE-days and {ce_d.percent_gt_50km:.1f}% of CE-days."
    )

    return {
        "data": d,
        "valid": valid,
        "eddies": eddies,
        "census": census,
        "daily": daily,
        "eddy_summary": eddy_summary,
        "sector": sector,
        "latitude": latitude,
        "sentences": sentences,
        "paragraph": paragraph,
    }

In [8]:
results = tilt_census_report(df_eddies)

display(results["census"].round(2))
display(results["daily"].round(2))
display(results["eddy_summary"].round(2))
display(results["sector"].round(2))
display(results["latitude"].round(2))

print("\nCOPYABLE RESULT SENTENCES\n")
print("\n".join(results["sentences"]))

print("\n\nCOPYABLE DRAFT PARAGRAPH\n")
display(Markdown(results["paragraph"]))

,unique_eddies,eddy_days,possible_tilt_days,valid_tilt_days,raw_coverage_percent,possible_coverage_percent
Cyc,,,,,,
All,2982,127426,109534,105621,82.89,96.43
AE,1446,64952,56276,53607,82.53,95.26
CE,1536,62474,53258,52014,83.26,97.66


,n,mean_km,median_km,q25_km,q75_km,q90_km,q95_km,q99_km,maximum_km,modal_5km_interval,percent_gt_10km,percent_gt_20km,percent_gt_50km,percent_gt_100km,median_tilt_over_Rc,percent_tilt_gt_Rc
Cyc,,,,,,,,,,,,,,,,
All,105621,23.87,17.66,9.53,31.57,50.39,65.16,100.29,266.95,5–10,73.45,44.48,10.18,1.01,0.23,6.26
AE,53607,26.08,19.61,10.49,34.75,54.74,70.21,106.14,266.95,5–10,76.63,49.11,12.40,1.29,0.25,6.65
CE,52014,21.58,16.01,8.71,28.30,45.33,59.30,91.97,202.39,5–10,70.18,39.71,7.90,0.72,0.22,5.87


,eddies,median_eddy_mean_km,median_eddy_median_km,median_eddy_minimum_km,median_eddy_maximum_km,median_eddy_range_km,median_mean_tilt_over_Rc,percent_min_below_10km,percent_max_above_50km,percent_max_above_100km,percent_max_above_200km,largest_eddy_maximum_km
Cyc,,,,,,,,,,,,
AE,1422,25.65,22.34,5.94,56.93,47.90,0.40,68.35,57.81,13.64,0.63,266.95
CE,1531,19.55,16.97,5.07,40.68,32.84,0.31,72.76,39.12,7.77,0.07,202.39


tilt_estimates  mean_tilt_km  median_tilt_km  \
Cyc Sector                                                          
AE  Downstream + S2           35634         21.19           15.71   
    Upstream + S1             17973         35.77           29.38   
CE  Downstream + S2           29459         16.49           12.03   
    Upstream + S1             22555         28.24           22.68   

                     q25_tilt_km  q75_tilt_km  median_tilt_over_Rc  \
Cyc Sector                                                           
AE  Downstream + S2         8.69        27.64                 0.20   
    Upstream + S1          17.23        47.25                 0.36   
CE  Downstream + S2         6.73        20.87                 0.17   
    Upstream + S1          13.52        37.26                 0.31   

                     percent_gt_50km  
Cyc Sector                            
AE  Downstream + S2             7.36  
    Upstream + S1              22.39  
CE  Downstream + S2             3.90  
    Upstream + S1              13.12

n  mean_tilt_km  median_tilt_km  q25_tilt_km  \
Cyc LatitudeZone                                                      
AE  32–36°S        16367         25.02           19.41        11.28   
    North of 32°S  14726         36.54           30.44        17.68   
    South of 36°S  22514         20.01           14.36         7.87   
CE  32–36°S        14393         18.72           13.60         7.98   
    North of 32°S  19965         29.02           23.45        14.04   
    South of 36°S  17656         15.51           11.48         6.22   

                   q75_tilt_km  
Cyc LatitudeZone                
AE  32–36°S              32.65  
    North of 32°S        48.55  
    South of 36°S        25.88  
CE  32–36°S              23.56  
    North of 32°S        38.18  
    South of 36°S        20.09


COPYABLE RESULT SENTENCES

Total eddies: 2,982.
AEs: 1,446; CEs: 1,536.
Total eddy-days: 127,426.
AE-days: 64,952; CE-days: 62,474.
Valid tilt estimates: 105,621 (82.9% of all eddy-days).
Possible-estimate success: 95.3% for AEs and 97.7% for CEs.
Mean AE tilt distance: 26.1 km.
Mean CE tilt distance: 21.6 km.
Median AE tilt distance: 19.6 km (IQR 10.5–34.8 km).
Median CE tilt distance: 16.0 km (IQR 8.7–28.3 km).
Modal 5-km tilt interval: 5–10 km for AEs and 5–10 km for CEs.
Tilt exceeded 50 km on 12.4% of AE-days and 7.9% of CE-days.
Tilt exceeded 100 km on 1.3% of AE-days and 0.7% of CE-days.
Median tilt/Rc: 0.25 for AEs and 0.22 for CEs.
Tilt exceeded one Rc on 6.6% of AE-days and 5.9% of CE-days.
Median per-eddy mean tilt: 25.7 km for AEs and 19.5 km for CEs.
At least one tilt above 50 km occurred in 57.8% of AEs and 39.1% of CEs.
At least one tilt above 100 km occurred in 13.6% of AEs and 7.8% of CEs.


COPYABLE DRAFT PARAGRAPH



Application of the Delta method produced 105,621 valid tilt estimates, corresponding to 82.9% of all eddy-days. The dataset contained 2,982 unique eddies, comprising 1,446 anticyclonic eddies and 1,536 cyclonic eddies. Valid estimates were obtained for 53,607 AE-days and 52,014 CE-days, representing 95.3% and 97.7% of possible estimates, respectively. Median tilt distances were 19.6 km for AEs and 16.0 km for CEs, with interquartile ranges of 10.5–34.8 km and 8.7–28.3 km, respectively. Tilt exceeded 50 km on 12.4% of AE-days and 7.9% of CE-days.

## Tilt Direction

In [9]:
from IPython.display import display

MIN_TILT_KM = 5.0
KEYS = ["Cyc", "Eddy"]

UPSTREAM = ["S1", "U1", "U2"]
DOWNSTREAM = ["S2", "D1", "D2"]

d = (
    df_eddies
    .dropna(subset=["TiltDir", "TiltDis"])
    .query("Cyc in ['AE', 'CE']")
    .query("TiltDis >= @MIN_TILT_KM")
    .copy()
)

theta = d["TiltDir"] % 360

# Compass sectors: 0° north, 90° east, 180° south, 270° west.
d["northward"] = (theta < 90) | (theta >= 270)
d["southward"] = (theta >= 90) & (theta < 270)
d["eastward"] = theta < 180
d["westward"] = theta >= 180

d["NE"] = (theta >= 0) & (theta < 90)
d["SE"] = (theta >= 90) & (theta < 180)
d["SW"] = (theta >= 180) & (theta < 270)
d["NW"] = (theta >= 270) & (theta < 360)

d["Sector"] = pd.NA
d.loc[d["Region"].isin(UPSTREAM), "Sector"] = "Upstream"
d.loc[d["Region"].isin(DOWNSTREAM), "Sector"] = "Downstream"


def circular_stats_deg(angles):
    angles = np.asarray(angles, float)
    angles = angles[np.isfinite(angles)]

    east = np.sin(np.deg2rad(angles)).mean()
    north = np.cos(np.deg2rad(angles)).mean()

    return (
        np.rad2deg(np.arctan2(east, north)) % 360,
        np.hypot(east, north),
    )


def direction_summary(g):
    mean_dir, concentration = circular_stats_deg(g["TiltDir"])

    # g.name is "AE"/"CE" for groupby("Cyc"), and a tuple such as
    # ("AE", "S1") for groupby(["Cyc", "Region"]).
    cyc = g.name[0] if isinstance(g.name, tuple) else g.name

    preferred_meridional = (
        g["northward"]
        if cyc == "AE"
        else g["southward"]
    )

    preferred_quadrant = (
        g["NE"]
        if cyc == "AE"
        else g["SE"]
    )

    return pd.Series({
        "n": len(g),
        "mean_direction_deg": mean_dir,
        "directional_concentration": concentration,

        "northward_percent":
            100 * g["northward"].mean(),

        "southward_percent":
            100 * g["southward"].mean(),

        "eastward_percent":
            100 * g["eastward"].mean(),

        "westward_percent":
            100 * g["westward"].mean(),

        "NE_percent_all_days":
            100 * g["NE"].mean(),

        "SE_percent_all_days":
            100 * g["SE"].mean(),

        "SW_percent_all_days":
            100 * g["SW"].mean(),

        "NW_percent_all_days":
            100 * g["NW"].mean(),

        "preferred_meridional_percent":
            100 * preferred_meridional.mean(),

        "preferred_quadrant_percent_all":
            100 * preferred_quadrant.mean(),

        "preferred_quadrant_given_meridional_percent":
            100 * preferred_quadrant[
                preferred_meridional
            ].mean(),

        "mean_tilt_km":
            g["TiltDis"].mean(),

        "median_tilt_km":
            g["TiltDis"].median(),

        "percent_tilt_gt_40km":
            100 * (g["TiltDis"] > 40).mean(),
    })


overall_direction = (
    d.groupby("Cyc")
    .apply(direction_summary, include_groups=False)
)

regional_direction = (
    d.dropna(subset=["Region"])
    .groupby(["Cyc", "Region"])
    .apply(direction_summary, include_groups=False)
)

sector_direction = (
    d.dropna(subset=["Sector"])
    .groupby(["Cyc", "Sector"])
    .apply(direction_summary, include_groups=False)
)

display(overall_direction.round(2))
display(regional_direction.round(2))
display(sector_direction.round(2))

,n,mean_direction_deg,directional_concentration,northward_percent,southward_percent,eastward_percent,westward_percent,NE_percent_all_days,SE_percent_all_days,SW_percent_all_days,NW_percent_all_days,preferred_meridional_percent,preferred_quadrant_percent_all,preferred_quadrant_given_meridional_percent,mean_tilt_km,median_tilt_km,percent_tilt_gt_40km
Cyc,,,,,,,,,,,,,,,,,
AE,49268.0,10.41,0.29,67.51,32.49,53.27,46.73,35.66,17.61,14.88,31.85,67.51,35.66,52.82,28.1,21.42,21.20
CE,46397.0,152.68,0.24,35.74,64.26,57.36,42.64,17.99,39.37,24.89,17.74,64.26,39.37,61.26,23.8,17.95,14.92


n  mean_direction_deg  directional_concentration  \
Cyc Region                                                           
AE  D1      20873.0                5.80                       0.26   
    D2       6751.0                9.65                       0.34   
    S1       2118.0               32.12                       0.21   
    S2       4188.0              331.33                       0.39   
    U1       6869.0               20.55                       0.30   
    U2       8469.0               30.11                       0.34   
CE  D1      13996.0              144.52                       0.34   
    D2       4987.0              146.87                       0.24   
    S1       3641.0              273.40                       0.30   
    S2       5798.0              254.39                       0.13   
    U1       7067.0              141.45                       0.51   
    U2      10908.0              148.25                       0.24   

            northward_percent  southward_percent  eastward_percent  \
Cyc Region                                                           
AE  D1                  66.41              33.59             51.84   
    D2                  70.64              29.36             53.99   
    S1                  59.96              40.04             56.99   
    S2                  70.68              29.32             35.70   
    U1                  66.91              33.09             56.84   
    U2                  68.53              31.47             61.11   
CE  D1                  31.67              68.33             62.15   
    D2                  37.84              62.16             59.51   
    S1                  52.62              47.38             30.40   
    S2                  46.31              53.69             41.76   
    U1                  23.55              76.45             71.18   
    U2                  36.62              63.38             58.58   

            westward_percent  NE_percent_all_days  SE_percent_all_days  \
Cyc Region                                                               
AE  D1                 48.16                34.83                17.01   
    D2                 46.01                37.62                16.37   
    S1                 43.01                31.73                25.26   
    S2                 64.30                21.47                14.23   
    U1                 43.16                38.43                18.40   
    U2                 38.89                41.89                19.21   
CE  D1                 37.85                18.24                43.91   
    D2                 40.49                21.84                37.68   
    S1                 69.60                11.59                18.81   
    S2                 58.24                14.88                26.87   
    U1                 28.82                16.33                54.85   
    U2                 41.42                20.77                37.81   

            SW_percent_all_days  NW_percent_all_days  \
Cyc Region                                             
AE  D1                    16.59                31.58   
    D2                    12.99                33.02   
    S1                    14.78                28.23   
    S2                    15.09                49.21   
    U1                    14.69                28.48   
    U2                    12.26                26.64   
CE  D1                    24.42                13.43   
    D2                    24.48                16.00   
    S1                    28.56                41.03   
    S2                    26.82                31.42   
    U1                    21.61                 7.22   
    U2                    25.57                15.85   

            preferred_meridional_percent  preferred_quadrant_percent_all  \
Cyc Region                                                                 
AE  D1                             66.41                           34.83   
    D2

n  mean_direction_deg  directional_concentration  \
Cyc Sector                                                               
AE  Downstream  31812.0                0.87                       0.29   
    Upstream    17456.0               26.67                       0.31   
CE  Downstream  24781.0              152.36                       0.23   
    Upstream    21616.0              153.01                       0.26   

                northward_percent  southward_percent  eastward_percent  \
Cyc Sector                                                               
AE  Downstream              67.87              32.13             50.17   
    Upstream                66.85              33.15             58.93   
CE  Downstream              36.34              63.66             56.85   
    Upstream                35.04              64.96             57.95   

                westward_percent  NE_percent_all_days  SE_percent_all_days  \
Cyc Sector                                                                   
AE  Downstream             49.83                33.66                16.51   
    Upstream               41.07                39.30                19.63   
CE  Downstream             43.15                18.18                38.67   
    Upstream               42.05                17.77                40.18   

                SW_percent_all_days  NW_percent_all_days  \
Cyc Sector                                                 
AE  Downstream                15.63                34.20   
    Upstream                  13.52                27.55   
CE  Downstream                24.99                18.16   
    Upstream                  24.78                17.27   

                preferred_meridional_percent  preferred_quadrant_percent_all  \
Cyc Sector                                                                     
AE  Downstream                         67.87                           33.66   
    Upstream                           66.85                           39.30   
CE  Downstream                         63.66                           38.67   
    Upstream                           64.96                           40.18   

                preferred_quadrant_given_meridional_percent  mean_tilt_km  \
Cyc Sector                                                                  
AE  Downstream                                        49.60         23.35   
    Upstream                                          58.78         36.74   
CE  Downstream                                        60.74         18.99   
    Upstream                                          61.85         29.32   

                median_tilt_km  percent_tilt_gt_40km  
Cyc Sector                                            
AE  Downstream           17.68                 13.99  
    Upstream             30.30                 34.34  
CE  Downstream           14.21                  8.19  
    Upstream             23.61                 22.64

PV gradient 

In [10]:
def angle_difference(a, b):
    """Signed circular difference a - b, in [-180°, 180°)."""
    return (a - b + 180) % 360 - 180


if "PV_grad_theta" in d.columns:
    # Hypothesis:
    # CE tilt follows the PV gradient.
    # AE tilt follows the direction opposite the PV gradient.
    d["expected_PV_tilt"] = np.where(
        d["Cyc"] == "AE",
        (d["PV_grad_theta"] + 180) % 360,
        d["PV_grad_theta"],
    )

    d["PV_offset"] = angle_difference(
        d["TiltDir"],
        d["expected_PV_tilt"],
    )

    def pv_summary(g):
        mean_offset, concentration = circular_stats_deg(
            g["PV_offset"] % 360
        )

        # Convert circular mean back to signed angle.
        mean_offset = angle_difference(mean_offset, 0)

        return pd.Series({
            "n": len(g),
            "mean_offset_deg": mean_offset,
            "alignment_concentration": concentration,
            "median_abs_offset_deg":
                g["PV_offset"].abs().median(),
            "within_30deg_percent":
                100 * (g["PV_offset"].abs() <= 30).mean(),
            "within_45deg_percent":
                100 * (g["PV_offset"].abs() <= 45).mean(),
            "within_90deg_percent":
                100 * (g["PV_offset"].abs() <= 90).mean(),
        })

    pv_alignment = (
        d.dropna(subset=["PV_grad_theta"])
        .groupby("Cyc")
        .apply(pv_summary, include_groups=False)
    )

    regional_pv_alignment = (
        d.dropna(subset=["PV_grad_theta", "Region"])
        .groupby(["Cyc", "Region"])
        .apply(pv_summary, include_groups=False)
    )

    display(pv_alignment.round(2))
    display(regional_pv_alignment.round(2))

In [11]:
ae = overall_direction.loc["AE"]
ce = overall_direction.loc["CE"]

sentences = [
    (
        f"Direction statistics used tilt estimates ≥ "
        f"{MIN_TILT_KM:.0f} km."
    ),
    (
        f"Equatorward tilt occurred on "
        f"{ae.preferred_meridional_percent:.1f}% of AE-days."
    ),
    (
        f"Poleward tilt occurred on "
        f"{ce.preferred_meridional_percent:.1f}% of CE-days."
    ),
    (
        f"Northeastward tilt occurred on "
        f"{ae.NE_percent_all_days:.1f}% of all AE-days."
    ),
    (
        f"Of the equatorward AE tilts, "
        f"{ae.preferred_quadrant_given_meridional_percent:.1f}% "
        f"were northeastward."
    ),
    (
        f"Southeastward tilt occurred on "
        f"{ce.SE_percent_all_days:.1f}% of all CE-days."
    ),
    (
        f"Of the poleward CE tilts, "
        f"{ce.preferred_quadrant_given_meridional_percent:.1f}% "
        f"were southeastward."
    ),
    (
        f"An eastward component occurred on "
        f"{ae.eastward_percent:.1f}% of AE-days and "
        f"{ce.eastward_percent:.1f}% of CE-days."
    ),
    (
        f"The circular-mean tilt bearing was "
        f"{ae.mean_direction_deg:.1f}° for AEs and "
        f"{ce.mean_direction_deg:.1f}° for CEs."
    ),
    (
        f"Directional concentration was "
        f"{ae.directional_concentration:.2f} for AEs and "
        f"{ce.directional_concentration:.2f} for CEs."
    ),
]

print("\n".join(sentences))

for cyc in ["AE", "CE"]:
    for sector in ["Upstream", "Downstream"]:
        r = sector_direction.loc[(cyc, sector)]

        print(
            f"{cyc} {sector.lower()}: "
            f"median tilt = {r.median_tilt_km:.1f} km; "
            f"tilt >40 km = {r.percent_tilt_gt_40km:.1f}%; "
            f"mean bearing = {r.mean_direction_deg:.1f}°; "
            f"concentration = {r.directional_concentration:.2f}."
        )

for (cyc, region), r in regional_direction.iterrows():
    print(
        f"{cyc} {region}: "
        f"northward = {r.northward_percent:.1f}%; "
        f"southward = {r.southward_percent:.1f}%; "
        f"eastward = {r.eastward_percent:.1f}%; "
        f"westward = {r.westward_percent:.1f}%; "
        f"median tilt = {r.median_tilt_km:.1f} km; "
        f"mean bearing = {r.mean_direction_deg:.1f}°."
    )

if "PV_grad_theta" in d.columns:
    for cyc, r in pv_alignment.iterrows():
        reference = (
            "opposite the PV gradient"
            if cyc == "AE"
            else "along the PV gradient"
        )

        print(
            f"{cyc} tilt relative to {reference}: "
            f"mean offset = {r.mean_offset_deg:.1f}°; "
            f"median absolute offset = "
            f"{r.median_abs_offset_deg:.1f}°; "
            f"{r.within_45deg_percent:.1f}% occurred within 45°; "
            f"alignment concentration = "
            f"{r.alignment_concentration:.2f}."
        )

Direction statistics used tilt estimates ≥ 5 km.
Equatorward tilt occurred on 67.5% of AE-days.
Poleward tilt occurred on 64.3% of CE-days.
Northeastward tilt occurred on 35.7% of all AE-days.
Of the equatorward AE tilts, 52.8% were northeastward.
Southeastward tilt occurred on 39.4% of all CE-days.
Of the poleward CE tilts, 61.3% were southeastward.
An eastward component occurred on 53.3% of AE-days and 57.4% of CE-days.
The circular-mean tilt bearing was 10.4° for AEs and 152.7° for CEs.
Directional concentration was 0.29 for AEs and 0.24 for CEs.
AE upstream: median tilt = 30.3 km; tilt >40 km = 34.3%; mean bearing = 26.7°; concentration = 0.31.
AE downstream: median tilt = 17.7 km; tilt >40 km = 14.0%; mean bearing = 0.9°; concentration = 0.29.
CE upstream: median tilt = 23.6 km; tilt >40 km = 22.6%; mean bearing = 153.0°; concentration = 0.26.
CE downstream: median tilt = 14.2 km; tilt >40 km = 8.2%; mean bearing = 152.4°; concentration = 0.23.
AE D1: northward = 66.4%; southward 